# Neural Network Regression Exercise (Real Data)

## Dataset: California Housing (scikit-learn)

**Goal:** predict `target` = median house value.

This dataset is fetched automatically (no manual download).

---


In [ ]:
# %% [code]
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, r2_score

import tensorflow as tf
import optuna

RANDOM_STATE: int = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)


## 1) Load data

In [ ]:
# %% [code]
def load_california_housing() -> pd.DataFrame:
    """Load the California housing dataset as a pandas DataFrame.

    Returns:
        DataFrame with features and a `target` column.
    """
    data = fetch_california_housing(as_frame=True)
    df = data.frame.copy()
    df.rename(columns={"MedHouseVal": "target"}, inplace=True)
    return df

df = load_california_housing()
df.head()


## 2) EDA

In [ ]:
# %% [code]
def eda_overview_regression(df: pd.DataFrame, target_col: str = "target") -> None:
    """Show basic EDA for a regression dataset."""
    print("Shape:", df.shape)
    print("\nMissing rate (top 10):")
    print(df.isna().mean().sort_values(ascending=False).head(10))
    print("\nDescribe:")
    display(df.describe().T)

    ax = df[target_col].hist(bins=40)
    ax.set_title(f"Target Distribution: {target_col}")
    plt.show()

    corr = df.corr(numeric_only=True)[target_col].sort_values(ascending=False)
    print("\nCorrelation with target:")
    print(corr)

eda_overview_regression(df)


## 3) Feature engineering + interaction features

We add a few ratio features and let Optuna decide whether polynomial interaction features (degree 1 vs 2) help.

In [ ]:
# %% [code]
def add_engineered_features_reg(df: pd.DataFrame) -> pd.DataFrame:
    """Add engineered ratio features for California housing.

    Args:
        df: Raw dataframe including base features and `target`.

    Returns:
        Copy with engineered features appended.
    """
    out = df.copy()
    eps = 1e-6
    out["RoomsPerOccup"] = out["AveRooms"] / (out["AveOccup"] + eps)
    out["BedrmsPerRoom"] = out["AveBedrms"] / (out["AveRooms"] + eps)
    out["PopPerOccup"] = out["Population"] / (out["AveOccup"] + eps)
    return out

df_fe = add_engineered_features_reg(df)
df_fe.head()


## 4) Split + preprocessing

In [ ]:
# %% [code]
@dataclass(frozen=True)
class RegressionConfig:
    """Config for the regression pipeline."""
    target: str = "target"

def make_preprocessor_reg(num_cols: list[str], poly_degree: int) -> ColumnTransformer:
    """Create numeric preprocessing with optional polynomial interaction expansion."""
    return ColumnTransformer([
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("poly", PolynomialFeatures(degree=poly_degree, include_bias=False)),
            ("scaler", StandardScaler()),
        ]), num_cols)
    ])

def to_dense(x: Any) -> np.ndarray:
    """Convert sparse matrix to dense numpy array if needed."""
    return x.toarray() if hasattr(x, "toarray") else np.asarray(x)

cfg = RegressionConfig()

X = df_fe.drop(columns=[cfg.target])
y = df_fe[cfg.target].astype(np.float32).values

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

num_cols = X.columns.tolist()
len(num_cols), num_cols[:5]


## 5) Keras model + Optuna (minimize RMSE)

In [ ]:
# %% [code]
def build_regressor(input_dim: int, n_layers: int, units: int, dropout: float, l2: float, lr: float) -> tf.keras.Model:
    """Build a simple MLP regressor in Keras.

    Args:
        input_dim: Number of input features.
        n_layers: Number of hidden layers.
        units: Units per hidden layer.
        dropout: Dropout rate.
        l2: L2 regularization strength.
        lr: Learning rate.

    Returns:
        Compiled Keras model.
    """
    inputs = tf.keras.Input(shape=(input_dim,))
    x = inputs
    for _ in range(n_layers):
        x = tf.keras.layers.Dense(
            units,
            activation="relu",
            kernel_regularizer=tf.keras.regularizers.l2(l2),
        )(x)
        x = tf.keras.layers.Dropout(dropout)(x)

    outputs = tf.keras.layers.Dense(1)(x)
    model = tf.keras.Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="mse",
        metrics=[tf.keras.metrics.RootMeanSquaredError(name="rmse")],
    )
    return model

def optuna_objective_reg(trial: optuna.Trial) -> float:
    """Optuna objective minimizing validation RMSE."""
    poly_degree = trial.suggest_int("poly_degree", 1, 2)
    n_layers = trial.suggest_int("n_layers", 1, 5)
    units = trial.suggest_int("units", 64, 512, step=64)
    dropout = trial.suggest_float("dropout", 0.0, 0.4)
    l2 = trial.suggest_float("l2", 1e-8, 1e-3, log=True)
    lr = trial.suggest_float("lr", 1e-4, 3e-3, log=True)
    batch_size = trial.suggest_categorical("batch_size", [64, 128, 256])

    pre = make_preprocessor_reg(num_cols, poly_degree)
    Xtr = to_dense(pre.fit_transform(X_train))
    Xva = to_dense(pre.transform(X_valid))

    model = build_regressor(Xtr.shape[1], n_layers, units, dropout, l2, lr)

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_rmse", mode="min", patience=10, restore_best_weights=True
        )
    ]
    try:
        from optuna.integration import TFKerasPruningCallback
        callbacks.append(TFKerasPruningCallback(trial, "val_rmse"))
    except Exception:
        pass

    model.fit(
        Xtr, y_train,
        validation_data=(Xva, y_valid),
        epochs=100,
        batch_size=batch_size,
        verbose=0,
        callbacks=callbacks,
    )

    pred = model.predict(Xva, verbose=0).ravel()
    rmse = mean_squared_error(y_valid, pred, squared=False)
    return float(rmse)


In [ ]:
# %% [code]
study = optuna.create_study(direction="minimize")
study.optimize(optuna_objective_reg, n_trials=40)

print("Best RMSE:", study.best_value)
print("Best params:", study.best_params)


## 6) Final retrain with best params + metrics

In [ ]:
# %% [code]
def train_best_and_evaluate_reg(best_params: Dict[str, Any]) -> Dict[str, float]:
    """Train a final regressor using best Optuna parameters and evaluate on validation."""
    pre = make_preprocessor_reg(num_cols, int(best_params["poly_degree"]))
    Xtr = to_dense(pre.fit_transform(X_train))
    Xva = to_dense(pre.transform(X_valid))

    model = build_regressor(
        input_dim=Xtr.shape[1],
        n_layers=int(best_params["n_layers"]),
        units=int(best_params["units"]),
        dropout=float(best_params["dropout"]),
        l2=float(best_params["l2"]),
        lr=float(best_params["lr"]),
    )

    model.fit(
        Xtr, y_train,
        validation_data=(Xva, y_valid),
        epochs=160,
        batch_size=int(best_params["batch_size"]),
        callbacks=[
            tf.keras.callbacks.EarlyStopping(
                monitor="val_rmse", mode="min", patience=15, restore_best_weights=True
            )
        ],
        verbose=0,
    )

    pred = model.predict(Xva, verbose=0).ravel()
    rmse = mean_squared_error(y_valid, pred, squared=False)
    r2 = r2_score(y_valid, pred)

    return {"val_rmse": float(rmse), "val_r2": float(r2)}

final_metrics = train_best_and_evaluate_reg(study.best_params)
final_metrics


## Next steps
- Try log-transforming the target.
- Add monotonic constraints (not native in Keras MLP, but can be approximated).
- Compare vs XGBoost/LightGBM.


## Summary: Baseline vs Tuned
This cell prints a **one‑glance comparison** between the baseline regressor and the Optuna‑tuned regressor.

In [ ]:

summary = {
    "baseline": baseline_metrics,
    "tuned": final_metrics,
    "rmse_reduction": baseline_metrics["baseline_val_rmse"] - final_metrics["val_rmse"],
    "r2_gain": final_metrics["val_r2"] - baseline_metrics["baseline_val_r2"],
}
summary
